# Clase 14 — POO Avanzada: Herencia, Polimorfismo y Métodos Mágicos

En esta clase exploraremos los conceptos avanzados de POO: herencia, polimorfismo, clases abstractas y métodos mágicos avanzados.

**Contenido de esta clase:**
1. Herencia
2. `super()` y MRO
3. Polimorfismo y duck typing
4. Herencia múltiple
5. Clases abstractas (ABC)
6. Métodos mágicos avanzados
7. `__slots__` para optimización
8. Ejercicio práctico


## 1. Herencia

La **herencia** permite crear una clase nueva (hija) que reutiliza y extiende una clase existente (padre).


In [ ]:
class Animal:
    """Clase base."""
    
    def __init__(self, nombre, edad):
        self.nombre = nombre
        self.edad = edad
    
    def presentarse(self):
        return f"Soy {self.nombre}, tengo {self.edad} años"
    
    def hacer_sonido(self):
        return "..."

class Perro(Animal):
    """Clase hija que hereda de Animal."""
    
    def __init__(self, nombre, edad, raza):
        super().__init__(nombre, edad)  # Llamar al constructor del padre
        self.raza = raza
    
    def hacer_sonido(self):  # Sobrescritura
        return "¡Guau!"

# Uso
mi_perro = Perro("Firulais", 5, "Labrador")
print(mi_perro.presentarse())
print(f"Raza: {mi_perro.raza}")
print(f"Sonido: {mi_perro.hacer_sonido()}")


In [ ]:
# isinstance() e issubclass()
print(f"¿mi_perro es Animal? {isinstance(mi_perro, Animal)}")
print(f"¿Perro es subclase de Animal? {issubclass(Perro, Animal)}")


## 2. `super()` y MRO (Method Resolution Order)

`super()` llama a métodos de la clase padre. MRO es el orden en que Python busca métodos cuando hay herencia múltiple.


In [ ]:
class A:
    def metodo(self):
        return "A"

class B(A):
    def metodo(self):
        return f"B -> {super().metodo()}"

class C(A):
    def metodo(self):
        return f"C -> {super().metodo()}"

class D(B, C):  # Herencia múltiple
    def metodo(self):
        return f"D -> {super().metodo()}"

# Ver el MRO
print("MRO de D:", [cls.__name__ for cls in D.__mro__])
print(f"d.metodo() = {D().metodo()}")


## 3. Polimorfismo y duck typing

**Polimorfismo**: el mismo método se comporta diferente según la clase.

**Duck typing**: "si camina como pato y suena como pato, es un pato". En Python, lo importante es qué métodos tiene un objeto, no su tipo.


In [ ]:
class Pato:
    def hablar(self):
        return "Cuac"

class Perro:
    def hablar(self):
        return "Guau"

class Gato:
    def hablar(self):
        return "Miau"

# Función polimórfica
def hacer_hablar(animal):
    """No importa el tipo, solo que tenga .hablar()"""
    print(animal.hablar())

# Funciona con cualquier objeto que tenga hablar()
hacer_hablar(Pato())
hacer_hablar(Perro())
hacer_hablar(Gato())


In [ ]:
# Duck typing en la práctica
class Pinguino:
    pass

# No tiene método hablar(), pero Python no verifica tipos
# hacer_hablar(Pinguino())  # AttributeError en tiempo de ejecución

# Mejor verificar con hasattr o try/except
def hacer_hablar_seguro(animal):
    try:
        print(animal.hablar())
    except AttributeError:
        print(f"{type(animal).__name__} no puede hablar")

hacer_hablar_seguro(Pinguino())


## 4. Herencia múltiple

Python permite que una clase herede de **múltiples** padres. Hay que tener cuidado con el MRO.


In [ ]:
class Volador:
    def volar(self):
        return "Estoy volando"

class Nadador:
    def nadar(self):
        return "Estoy nadando"

class Pato(Volador, Nadador):
    """Pato que vuela y nada."""
    pass

# Uso
pato = Pato()
print(pato.volar())
print(pato.nadar())


In [ ]:
# Mixin pattern: clases pequeñas con funcionalidad específica
class JSONSerializableMixin:
    """Agrega capacidad de serializar a JSON."""
    def to_json(self):
        import json
        return json.dumps(self.__dict__, default=str)

class LoggerMixin:
    """Agrega logging."""
    def log(self, mensaje):
        print(f"[{type(self).__name__}] {mensaje}")

class Usuario(LoggerMixin, JSONSerializableMixin):
    def __init__(self, nombre, email):
        self.nombre = nombre
        self.email = email

# Uso
u = Usuario("Ana", "ana@uni.edu")
u.log("Usuario creado")
print(u.to_json())


## 5. Clases abstractas (ABC)

Las **clases abstractas** definen una interfaz que las clases hijas deben implementar.


In [ ]:
from abc import ABC, abstractmethod

class Figura(ABC):
    """Clase abstracta para figuras geométricas."""
    
    @abstractmethod
    def area(self):
        """Calcula el área. Debe ser implementado por las hijas."""
        pass
    
    @abstractmethod
    def perimetro(self):
        """Calcula el perímetro. Debe ser implementado por las hijas."""
        pass
    
    def descripcion(self):
        return f"Soy una {type(self).__name__}"

class Circulo(Figura):
    def __init__(self, radio):
        self.radio = radio
    
    def area(self):
        import math
        return math.pi * self.radio ** 2
    
    def perimetro(self):
        import math
        return 2 * math.pi * self.radio

# Uso
c = Circulo(5)
print(f"Área: {c.area():.2f}")
print(f"Perímetro: {c.perimetro():.2f}")
print(c.descripcion())

# No se puede instanciar una clase abstracta
# fig = Figura()  # TypeError


## 6. Métodos mágicos avanzados


In [ ]:
class Carrito:
    """Carrito de compras con métodos mágicos."""
    
    def __init__(self):
        self.items = []
    
    def agregar(self, producto, precio):
        self.items.append((producto, precio))
    
    def __len__(self):
        """len(carrito) retorna número de items."""
        return len(self.items)
    
    def __getitem__(self, indice):
        """carrito[0] para acceder a items."""
        return self.items[indice]
    
    def __iter__(self):
        """Permite usar for sobre el carrito."""
        return iter(self.items)
    
    def __contains__(self, producto):
        """Permite usar 'in'."""
        return any(p == producto for p, _ in self.items)
    
    def __bool__(self):
        """Permite usar if carrito:"""
        return len(self.items) > 0
    
    def __add__(self, otro):
        """Concatenar carritos con +."""
        nuevo = Carrito()
        nuevo.items = self.items + otro.items
        return nuevo

# Uso
c1 = Carrito()
c1.agregar("Laptop", 1500)
c1.agregar("Mouse", 25)

c2 = Carrito()
c2.agregar("Teclado", 75)

print(f"Items en c1: {len(c1)}")
print(f"Primer item: {c1[0]}")
print(f"¿'Mouse' in c1? {'Mouse' in c1}")
print(f"¿Carrito vacío? {bool(Carrito())}")
print(f"¿Carrito con items? {bool(c1)}")

# Iterar
print("\nItems:")
for producto, precio in c1:
    print(f"  {producto}: ${precio}")

# Sumar carritos
c3 = c1 + c2
print(f"\nCarrito combinado tiene {len(c3)} items")


## 7. `__slots__` para optimización

`__slots__` limita los atributos permitidos, ahorrando memoria.


In [ ]:
import sys

class PuntoNormal:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PuntoOptimizado:
    __slots__ = ['x', 'y']  # Solo estos atributos son permitidos
    
    def __init__(self, x, y):
        self.x = x
        self.y = y

# Comparar uso de memoria
p1 = PuntoNormal(1, 2)
p2 = PuntoOptimizado(1, 2)

print(f"PuntoNormal: {sys.getsizeof(p1)} bytes")
print(f"PuntoOptimizado: {sys.getsizeof(p2)} bytes")

# Intentar agregar atributo no declarado
try:
    p2.z = 3
except AttributeError as e:
    print(f"\nError: {e}")


## 8. Ejercicio Práctico: Librería de Vectores Matemáticos

**Contexto:** Vas a crear una clase `Vector` que soporte operaciones matemáticas vectoriales usando herencia y métodos mágicos.

**Instrucciones:**

Crea un archivo `vector.py` con:

1. Clase `Vector2D` con `x` e `y`, métodos:
   - `__repr__`, `__str__`, `__eq__`
   - `__add__`, `__sub__`, `__mul__` (escalar)
   - `__abs__` (magnitud)
   - `__len__` (retorna 2)

2. Clase `Vector3D` que herede de `Vector2D`, agregue `z`, y sobrescriba:
   - `__len__` (retorna 3)
   - `__repr__`

**Pistas:**
- Usa `super().__init__()` para inicializar `x` e `y` en `Vector3D`.
- La magnitud es `sqrt(x² + y²)` (o `+ z²` para 3D).
- `__mul__` debe aceptar un escalar: `v * 2` o `2 * v`.

**Restricción del ejercicio:**
- No uses `numpy`.
- Usa solo el módulo `math` para cálculos.


In [ ]:
"""
Solución al Ejercicio Práctico
Este código debe ir en un archivo vector.py
"""

import math

class Vector2D:
    """Vector en 2 dimensiones."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Vector2D(x={self.x}, y={self.y})"
    
    def __str__(self):
        return f"({self.x}, {self.y})"
    
    def __eq__(self, otro):
        return self.x == otro.x and self.y == otro.y
    
    def __add__(self, otro):
        return Vector2D(self.x + otro.x, self.y + otro.y)
    
    def __sub__(self, otro):
        return Vector2D(self.x - otro.x, self.y - otro.y)
    
    def __mul__(self, escalar):
        return Vector2D(self.x * escalar, self.y * escalar)
    
    def __rmul__(self, escalar):
        return self.__mul__(escalar)
    
    def __abs__(self):
        return math.sqrt(self.x ** 2 + self.y ** 2)
    
    def __len__(self):
        return 2

class Vector3D(Vector2D):
    """Vector en 3 dimensiones."""
    
    def __init__(self, x, y, z):
        super().__init__(x, y)
        self.z = z
    
    def __repr__(self):
        return f"Vector3D(x={self.x}, y={self.y}, z={self.z})"
    
    def __str__(self):
        return f"({self.x}, {self.y}, {self.z})"
    
    def __add__(self, otro):
        return Vector3D(self.x + otro.x, self.y + otro.y, self.z + otro.z)
    
    def __abs__(self):
        return math.sqrt(self.x ** 2 + self.y ** 2 + self.z ** 2)
    
    def __len__(self):
        return 3


In [ ]:
# Probar Vector2D
v1 = Vector2D(3, 4)
v2 = Vector2D(1, 2)
print(f"v1: {v1}")
print(f"v1 + v2: {v1 + v2}")
print(f"v1 * 3: {v1 * 3}")
print(f"|v1|: {abs(v1):.2f}")
print(f"len(v1): {len(v1)}")

# Probar Vector3D
v3 = Vector3D(1, 2, 3)
v4 = Vector3D(4, 5, 6)
print(f"\nv3: {v3}")
print(f"v3 + v4: {v3 + v4}")
print(f"|v3|: {abs(v3):.2f}")
print(f"len(v3): {len(v3)}")

# Polimorfismo: misma función, diferentes tipos
def describir_vector(v):
    print(f"Vector de {len(v)}D: {v}, magnitud: {abs(v):.2f}")

describir_vector(v1)
describir_vector(v3)


## Resumen de la clase

En esta clase aprendimos:

1. **Herencia:** crear clases hijas que reutilizan código del padre.
2. **`super()` y MRO:** llamar al padre y resolver el orden de herencia múltiple.
3. **Polimorfismo y duck typing:** mismo método, diferentes comportamientos.
4. **Herencia múltiple:** una clase puede heredar de varios padres.
5. **Clases abstractas (ABC):** definir interfaces obligatorias.
6. **Métodos mágicos:** `__len__`, `__getitem__`, `__iter__`, `__contains__`, `__bool__`, `__add__`, etc.
7. **`__slots__`:** optimizar memoria limitando atributos.

**En la siguiente clase** (Clase 15) veremos **Programación Funcional**: funciones puras, `map`/`filter`/`reduce`, comprehensions y composición.
